# 06 Compartir oro

Publica las cuatro tablas de oro en el share `oceanwatch_eq8_oro` para el bono de Delta Sharing. No forma parte del job.

`CREATE SHARE` pide un privilegio de metastore que `mine-group8` no tiene. Esta celda falla con `PERMISSION_DENIED` hasta que un administrador cree el share y el destinatario, o conceda ese privilegio. El alias dentro del share sigue siendo `oro`.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))


In [ ]:
%run ./_config


## Bono · Proveedor de Open Sharing
Este modo publica las cuatro tablas Delta del contrato bajo el esquema compartido `oro`. Un destinatario de token se activa mediante un archivo de credenciales de un solo receptor. La firma de control del día se puede comparar en el notebook Free Edition. Ejecutar con permisos sobre el share y sus tablas.


In [ ]:
SHARE = "oceanwatch_eq8_oro"
RECIPIENT = "oceanwatch_eq8_free"

def publicar_share():
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.sharing import AuthenticationType
    w = WorkspaceClient()
    spark.sql(f"CREATE SHARE IF NOT EXISTS {SHARE} COMMENT 'Productos diarios OceanWatch del equipo 8'")
    # Sin sharing identifier se crea un destinatario OPEN de token y archivo .share.
    spark.sql(f"CREATE RECIPIENT IF NOT EXISTS {RECIPIENT} COMMENT 'Receptor independiente Free Edition'")
    receptor = w.recipients.get(RECIPIENT)
    if receptor.authentication_type != AuthenticationType.TOKEN:
        raise ValueError("El destinatario existe con otro mecanismo. Se requiere Open Sharing con token.")
    inventario = {r['name']: r['shared_object'] for r in spark.sql(f"SHOW ALL IN SHARE {SHARE}").collect()}
    esperados = {}
    for tabla in OROS:
        fisico = tabla.replace("`","")
        alias = "oro." + fisico.rsplit(".", 1)[-1]
        esperados[alias] = fisico
    if any(k not in esperados or v != esperados[k] for k,v in inventario.items()):
        raise ValueError("El share existente contiene un objeto fuera del contrato del equipo.")
    for alias, fisico in esperados.items():
        if alias not in inventario:
            spark.sql(f"ALTER SHARE {SHARE} ADD TABLE {fisico} AS {alias}")
    spark.sql(f"GRANT SELECT ON SHARE {SHARE} TO RECIPIENT {RECIPIENT}")
    display(spark.sql(f"SHOW ALL IN SHARE {SHARE}"))
    display(spark.sql(f"SHOW GRANTS ON SHARE {SHARE}"))
    print("Catalog Explorer → OpenSharing / Delta Sharing → Shared by me → Recipients →",
          RECIPIENT, "→ Activation link. El receptor descarga el perfil de uso privado.")


In [ ]:
def firma_diaria(tablas, dia):
    import datetime
    from databricks.sdk import WorkspaceClient
    iso = datetime.date.fromisoformat(str(dia)).isoformat()
    resultado = {}
    for tabla in tablas:
        # Misma receta en ambos workspaces. Las cuatro huellas solo contienen agregados.
        cols = spark.table(tabla).columns
        campos = ",".join("`"+c.replace("`","``")+"`" for c in cols)
        f = spark.sql(f"""SELECT count(*) n, coalesce(sum(cast(xxhash64({campos}) as decimal(38,0))),0) h
                        FROM {tabla} WHERE fecha=DATE '{iso}'""").first()
        resultado[tabla.replace("`","").split(".")[-1]] = {"filas":int(f.n),"suma_hash":str(f.h)}
    return {"workspace":WorkspaceClient().config.host.rstrip("/"), "fecha":iso, "tablas":resultado}


In [ ]:
import json
OROS = [GOLD_TRAY, GOLD_PUERTO, GOLD_TRAFICO, GOLD_ANOM]
publicar_share()
print(json.dumps(firma_diaria(OROS, fecha),ensure_ascii=False))
